# 한국어 토크나이저 벤치마크

## 분석 목표

| # | 내용 |
|---|------|
| 1 | BPE와 WordPiece 서브워드 토크나이저를 학습 데이터 크기별로 비교한다 |
| 2 | Fertility, UNK Rate, 속도 지표로 모든 토크나이저를 종합 평가한다 |

## 실행 순서

1. `data_crawling.ipynb` 를 먼저 실행하여 `data/naver_news_collected.csv` 를 생성한다.
2. 이 노트북을 위에서부터 순서대로 실행한다.

In [ ]:
import subprocess, sys, os, platform

def setup_java():
    """KoNLPy 사용을 위해 Java를 설치하고 JAVA_HOME을 설정한다."""
    if platform.system() == "Darwin":
        result = subprocess.run(["/usr/libexec/java_home"], capture_output=True)
        if result.returncode != 0:
            print("Java를 설치한다 (brew install openjdk)...")
            subprocess.run(["brew", "install", "openjdk"], check=True)
            java_home = "/opt/homebrew/opt/openjdk"
        else:
            java_home = result.stdout.decode().strip()
    else:
        result = subprocess.run(["which", "java"], capture_output=True)
        if result.returncode != 0:
            subprocess.run(["apt-get", "install", "-y", "default-jdk"], check=True)
        java_home = subprocess.run(
            "dirname $(dirname $(readlink -f $(which java)))",
            shell=True, capture_output=True
        ).stdout.decode().strip()
    os.environ["JAVA_HOME"] = java_home
    print(f"JAVA_HOME: {java_home}")

setup_java()

# 필요 패키지를 설치한다.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "tokenizers", "JPype1", "konlpy", "python-mecab-ko",
    "matplotlib", "seaborn", "pandas"], check=True)
print("패키지 설치 완료")

In [ ]:
# 라이브러리를 불러오고 전역 설정을 초기화한다.
import os, time, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from collections import Counter
from tokenizers import Tokenizer
from tokenizers.models import BPE, WordPiece
from tokenizers.trainers import BpeTrainer, WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

warnings.filterwarnings("ignore")

# 한글 고딕 폰트를 설정한다.
for font_name in ["AppleGothic", "NanumGothic", "Nanum Gothic", "Malgun Gothic"]:
    if any(font_name in f.name for f in fm.fontManager.ttflist):
        plt.rcParams["font.family"] = font_name
        print(f"폰트 설정: {font_name}")
        break
plt.rcParams["axes.unicode_minus"] = False

# 출력 디렉토리를 생성한다.
for d in ["data", "figures", "results"]:
    os.makedirs(d, exist_ok=True)

# 하이퍼파라미터를 설정한다.
VOCAB_SIZE   = 2000
SEED         = 42
TRAIN_RATIOS = [0.1, 0.5, 1.0]  # 학습 데이터 비율: 10%, 50%, 100%

print(f"VOCAB_SIZE={VOCAB_SIZE}, SEED={SEED}, TRAIN_RATIOS={TRAIN_RATIOS}")

## 1. 데이터 준비

`data/naver_news_collected.csv`에서 뉴스 데이터를 로드하고 학습/평가 셋으로 분리한다.

In [ ]:
# 뉴스 데이터를 로드하고 학습/평가 셋으로 분리한다.

CSV_PATH = "data/naver_news_collected.csv"
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"{CSV_PATH} 가 없습니다. data_crawling.ipynb 를 먼저 실행하세요.")

df = pd.read_csv(CSV_PATH).dropna(subset=["description"]).drop_duplicates(subset=["description"])
texts = df["description"].astype(str).tolist()
print(f"전체 문장 수: {len(texts):,}")

# 학습/평가 셋을 분리한다 (평가셋 ≈ 1%).
np.random.seed(SEED)
idx = np.random.permutation(len(texts))
n_test = max(1, int(len(texts) * 0.01))
test_texts  = [texts[i] for i in idx[:n_test]]
train_texts = [texts[i] for i in idx[n_test:]]

train_bytes = sum(len(t.encode()) for t in train_texts)
test_bytes  = sum(len(t.encode()) for t in test_texts)
print(f"학습 문장: {len(train_texts):,}  ({train_bytes/1024:.1f} KB)")
print(f"평가 문장: {len(test_texts):,}  ({test_bytes/1024:.1f} KB)")
print(f"학습 단어 수: {sum(len(t.split()) for t in train_texts):,}")

assert train_bytes >= 1024 * 1024, "학습 데이터가 1 MB 미만입니다."
assert test_bytes >= 2048, f"평가 데이터가 2 KB 미만입니다: {test_bytes} bytes"
print("✅ 데이터 조건 충족")

In [ ]:
# 학습 데이터 분포를 시각화한다.

lengths = [len(t) for t in train_texts]

# (a) 문장 길이 분포
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("문장 길이 (문자 수)")
ax.set_ylabel("빈도")
ax.set_title("학습 데이터 문장 길이 분포")
plt.tight_layout()
plt.savefig("figures/fig_sent_length.png", dpi=150)
plt.show()

# (b) 상위 20개 단어 빈도를 출력한다.
stopwords = {"의", "을", "를", "이", "가", "은", "는", "과", "와", "에", "에서", "로", "으로", "도", "만", "하"}
word_counts = Counter(w for t in train_texts for w in t.split() if w not in stopwords and len(w) > 1)
top_words = word_counts.most_common(20)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([w for w, _ in top_words[::-1]], [c for _, c in top_words[::-1]], color="coral")
ax.set_xlabel("빈도")
ax.set_title("상위 20개 단어")
plt.tight_layout()
plt.savefig("figures/fig_top_words.png", dpi=150)
plt.show()

print(f"전체 어휘 크기: {len(word_counts):,}")

## 2. 토크나이저 구현

| 토크나이저 | 방식 | 특징 |
|-----------|------|------|
| **BPE** | 서브워드 | 절대 빈도 `freq(AB)` 기반으로 병합 규칙을 학습한다 |
| **WordPiece** | 서브워드 | 상호정보량 `freq(AB)/(freq(A)×freq(B))` 기반으로 병합한다 |
| **Mecab** | 형태소 | 속도가 가장 빠르며 품사 태깅을 지원한다 |
| **Okt** | 형태소 | 구어체·SNS 텍스트에 강하며 정규화 기능을 제공한다 |
| **Kkma** | 형태소 | 세밀한 형태소 분석이 가능하나 속도가 느리다 |

In [ ]:
# 토크나이저 클래스를 정의한다.
import tempfile

class SubwordTokenizer:
    """HuggingFace tokenizers 라이브러리를 래핑하는 서브워드 토크나이저 기반 클래스."""

    def __init__(self, model_type: str, vocab_size: int):
        self.vocab_size = vocab_size
        self.model_type = model_type
        if model_type == "bpe":
            self.tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
            self.trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"])
        else:
            self.tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
            self.trainer = WordPieceTrainer(vocab_size=vocab_size, special_tokens=["[UNK]"])
        self.tokenizer.pre_tokenizer = Whitespace()

    def train(self, texts: list):
        """텍스트 리스트로 토크나이저를 학습한다."""
        with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8") as f:
            f.write("\n".join(texts))
            tmp_path = f.name
        self.tokenizer.train([tmp_path], self.trainer)
        os.unlink(tmp_path)

    def tokenize(self, text: str) -> list:
        return self.tokenizer.encode(text).tokens

    def __repr__(self):
        return f"{self.model_type.upper()}(vocab={self.vocab_size})"


class KonlpyTokenizer:
    """KoNLPy 형태소 분석기(Okt/Kkma)를 래핑하는 클래스."""

    def __init__(self, analyzer_name: str):
        self.name = analyzer_name
        if analyzer_name == "Okt":
            from konlpy.tag import Okt
            self._tagger = Okt()
        elif analyzer_name == "Kkma":
            from konlpy.tag import Kkma
            self._tagger = Kkma()
        else:
            raise ValueError(f"지원하지 않는 분석기: {analyzer_name}")

    def tokenize(self, text: str) -> list:
        return self._tagger.morphs(text)

    def __repr__(self):
        return f"KoNLPy({self.name})"


class MecabTokenizer:
    """python-mecab-ko 패키지를 래핑하는 MeCab 형태소 분석기 클래스."""

    def __init__(self):
        from mecab import MeCab
        self._tagger = MeCab()
        self.name = "Mecab"

    def tokenize(self, text: str) -> list:
        return [token.surface for token in self._tagger.parse(text)]

    def __repr__(self):
        return "Mecab"


# 형태소 분석기를 초기화한다.
morph_tokenizers = {}
for name, cls, args in [
    ("Mecab", MecabTokenizer, []),
    ("Okt",   KonlpyTokenizer, ["Okt"]),
    ("Kkma",  KonlpyTokenizer, ["Kkma"]),
]:
    try:
        morph_tokenizers[name] = cls(*args)
        print(f"✅ {name} 로드 성공")
    except Exception as e:
        print(f"❌ {name} 로드 실패: {e}")

print(f"\n사용 가능한 형태소 분석기: {list(morph_tokenizers.keys())}")

## 3. 학습 및 평가

Fertility, UNK Rate, 속도를 측정하여 토크나이저 성능을 정량 평가한다.

| 지표 | 설명 |
|------|------|
| **Fertility** | 토큰 수 / 공백 기준 단어 수 — 낮을수록 압축 효율이 높다 |
| **UNK Rate** | `[UNK]` 토큰 비율 — 낮을수록 미등록어 처리 능력이 높다 |
| **Speed (ms)** | 평가 문장 전체 토크나이징 소요 시간 |

In [ ]:
# 평가 함수를 정의하고 BPE / WordPiece를 학습 데이터 비율별로 평가한다.

# 비교에 사용할 지표 목록을 전역으로 정의한다.
METRICS_INFO = [
    ("fertility", "Fertility",  "토큰 수 / 단어 수"),
    ("unk_rate",  "UNK Rate",   "미등록 토큰 비율"),
    ("speed_ms",  "Speed (ms)", "토크나이징 소요 시간"),
]

def evaluate(tokenizer, texts: list) -> dict:
    """Fertility, UNK Rate, Speed를 측정한다."""
    all_tokens, total_words = [], 0
    t0 = time.perf_counter()
    for text in texts:
        tokens = tokenizer.tokenize(text)
        all_tokens.extend(tokens)
        total_words += len(text.split())
    elapsed = (time.perf_counter() - t0) * 1000  # ms
    unk_count = sum(1 for t in all_tokens if t == "[UNK]")
    fertility = len(all_tokens) / total_words if total_words > 0 else 0
    unk_rate  = unk_count / len(all_tokens) if all_tokens else 0
    return {"fertility": round(fertility, 4), "unk_rate": round(unk_rate, 4), "speed_ms": round(elapsed, 2)}

# 학습 데이터 비율을 변경하며 성능을 측정한다. 100% 모델은 이후 예시에서 재사용한다.
records = []
final_models = {}  # ratio=1.0 학습 완료 모델을 보관한다.
for ratio in TRAIN_RATIOS:
    n = max(1, int(len(train_texts) * ratio))
    subset = train_texts[:n]
    for model_type in ["bpe", "wordpiece"]:
        tok = SubwordTokenizer(model_type, VOCAB_SIZE)
        tok.train(subset)
        metrics = evaluate(tok, test_texts)
        records.append({"model": model_type.upper(), "ratio": ratio, **metrics})
        print(f"[{model_type.upper():10s}] ratio={ratio:.2f}  {metrics}")
        if ratio == 1.0:
            final_models[model_type.upper()] = tok  # 100% 모델을 저장한다.

df_subword = pd.DataFrame(records)
print("\nBPE vs WordPiece 결과:")
print(df_subword.to_string(index=False))

## 4. BPE vs WordPiece 학습 비교

학습 데이터 비율을 10% → 100%로 늘리면서 두 알고리즘의 성능 변화를 분석한다.

In [ ]:
# BPE vs WordPiece 비교 결과를 메트릭별로 시각화한다.

style = {
    "BPE":       dict(linestyle="--", marker="s", color="tomato"),
    "WORDPIECE": dict(linestyle="-",  marker="o", color="steelblue"),
}

for col, title, ylabel in METRICS_INFO:
    fig, ax = plt.subplots(figsize=(7, 4))
    for model, grp in df_subword.groupby("model"):
        ax.plot(grp["ratio"] * 100, grp[col], label=model, **style.get(model, {}))
        for _, row in grp.iterrows():
            offset = 8 if model == "WORDPIECE" else -14
            ax.annotate(f"{row[col]:.3f}",
                        (row["ratio"] * 100, row[col]),
                        textcoords="offset points", xytext=(0, offset),
                        ha="center", fontsize=8)
    ax.set_xlabel("학습 데이터 비율 (%)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"BPE vs WordPiece — {title}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"figures/fig_subword_{col}.png", dpi=150)
    plt.show()

## 5. 종합 비교

BPE, WordPiece(100% 학습)와 형태소 분석기(Mecab, Okt, Kkma)를 세 가지 관점에서 비교한다.

- **5-A. 정량 비교**: Fertility, UNK Rate, Speed 지표 비교
- **5-B. 예시 문장 토크나이징**: 동일 문장에 대한 각 토크나이저의 분리 결과 및 토큰 수 비교
- **5-C. UNK 단어 비교**: 토크나이저별 UNK 단어 비교

In [ ]:
# 5-A. BPE, WordPiece(100%)와 형태소 분석기를 정량 지표로 비교하고 결과를 저장한다.

# 형태소 분석기를 평가한다.
morph_records = []
for name, tok in morph_tokenizers.items():
    metrics = evaluate(tok, test_texts)
    morph_records.append({"model": name, **metrics})
    print(f"[{name:6s}] {metrics}")
df_morph = pd.DataFrame(morph_records)

# 서브워드 100% 결과와 형태소 분석기 결과를 합친다.
subword_100 = df_subword[df_subword["ratio"] == 1.0][["model", "fertility", "unk_rate", "speed_ms"]].copy()
df_combined = pd.concat([subword_100, df_morph[["model", "fertility", "unk_rate", "speed_ms"]]], ignore_index=True)

print("\n종합 비교 결과:")
print(df_combined.to_string(index=False))

# 종합 비교 차트를 그린다 (모델 순서: BPE, WORDPIECE, Mecab, Okt, Kkma).
colors = ["tomato", "steelblue", "#4C72B0", "#DD8452", "#55A868"]
for col, title, ylabel in METRICS_INFO:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(df_combined["model"], df_combined[col], color=colors[:len(df_combined)])
    for i, row in df_combined.iterrows():
        ax.text(i, row[col] + row[col] * 0.02, f"{row[col]:.3f}", ha="center", fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(f"전체 토크나이저 종합 비교 — {title}")
    plt.tight_layout()
    plt.savefig(f"figures/fig_combined_{col}.png", dpi=150)
    plt.show()

# 결과를 CSV로 저장한다.
df_subword.to_csv("results/subword_results.csv", index=False, encoding="utf-8-sig")
df_morph.to_csv("results/morph_results.csv", index=False, encoding="utf-8-sig")
df_combined.to_csv("results/combined_results.csv", index=False, encoding="utf-8-sig")
print("\n결과 저장 완료: results/")

### 5-B. 예시 문장 토크나이징

동일한 예시 문장에 각 토크나이저를 적용하여 토큰 분리 결과와 토큰 수를 정성적으로 비교한다.

In [ ]:
# 예시 문장에 대해 각 토크나이저의 분리 결과와 토큰 수를 비교한다.

EXAMPLE_SENTS = [
    "미국 연방준비제도가 기준금리를 0.25%포인트 인상했다.",
    "비트코인 가격이 급등하며 가상화폐 시장 전체가 강세를 보였다.",
    "서울 강남 아파트 전세가가 2억 원 이상 하락했다는 분석이 나왔다.",
]

# BPE/WordPiece(100%)와 형태소 분석기를 하나의 딕셔너리로 합친다.
all_tokenizers = {**final_models, **morph_tokenizers}

# 문장별로 각 토크나이저의 분리 결과를 출력한다.
for sent in EXAMPLE_SENTS:
    print(f"\n{'─'*65}")
    print(f"문장: {sent}")
    print(f"{'─'*65}")
    rows = []
    for name, tok in all_tokenizers.items():
        tokens = tok.tokenize(sent)
        rows.append({"토크나이저": name, "토큰 수": len(tokens), "토큰 목록": " | ".join(tokens)})
    print(pd.DataFrame(rows).to_string(index=False))

# 3개 문장 평균 토큰 수를 막대 차트로 시각화한다.
avg_counts = {
    name: sum(len(tok.tokenize(s)) for s in EXAMPLE_SENTS) / len(EXAMPLE_SENTS)
    for name, tok in all_tokenizers.items()
}
colors = ["tomato", "steelblue", "#4C72B0", "#DD8452", "#55A868"]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(avg_counts.keys(), avg_counts.values(), color=colors[:len(avg_counts)])
for bar, val in zip(bars, avg_counts.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{val:.1f}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("평균 토큰 수")
ax.set_title("예시 문장 3개 기준 — 토크나이저별 평균 토큰 수")
plt.tight_layout()
plt.savefig("figures/fig_example_token_count.png", dpi=150)
plt.show()

### 5-C. UNK 단어 분석

평가 데이터에서 각 토크나이저가 미등록어로 처리한 단어를 수집하여 빈도 순으로 시각화한다.

| 토크나이저 | UNK 판별 기준 |
|-----------|--------------|
| BPE · WordPiece | 출력 토큰에 `[UNK]` 포함 |
| Mecab | POS 태그가 `SL`(외국어) · `SW`(특수문자) · `UNKNOWN` |
| Okt | POS 태그가 `Alpha` · `Foreign` · `Unknown` |
| Kkma | POS 태그가 `UN`(미등록어) |

In [ ]:
# 각 토크나이저의 UNK 단어를 수집하고 빈도 순으로 시각화한다.

TOP_N = 20

# 서브워드 토크나이저: 출력 토큰에 [UNK]가 포함된 단어를 UNK로 간주한다.
def subword_unk_words(tok, texts):
    counter = Counter()
    for text in texts:
        for word in text.split():
            if "[UNK]" in tok.tokenize(word):
                counter[word] += 1
    return counter

# 형태소 분석기: POS 태그 기반으로 미등록어를 판별한다.
MORPH_UNK_TAGS = {
    "Mecab": {"SL", "SW", "UNKNOWN"},
    "Okt":   {"Alpha", "Foreign", "Unknown"},
    "Kkma":  {"UN"},
}

def morph_unk_words(name, tagger, texts):
    counter = Counter()
    unk_tags = MORPH_UNK_TAGS.get(name, set())
    for text in texts:
        pos_result = tagger.pos(text)  # [(형태소, POS), ...]
        for morph, pos in pos_result:
            if pos in unk_tags:
                counter[morph] += 1
    return counter

# 전체 토크나이저의 UNK 단어를 수집한다.
unk_counters = {}

for name, tok in final_models.items():
    unk_counters[name] = subword_unk_words(tok, test_texts)

for name, tok in morph_tokenizers.items():
    unk_counters[name] = morph_unk_words(name, tok._tagger, test_texts)

# 결과 요약을 출력한다.
for name, counter in unk_counters.items():
    print(f"[{name:10s}] 고유 UNK 단어: {len(counter):,}개  |  상위 5개: {counter.most_common(5)}")

# 토크나이저별 UNK 상위 단어를 개별 차트로 시각화한다.
colors = {"BPE": "tomato", "WORDPIECE": "steelblue",
          "Mecab": "#4C72B0", "Okt": "#DD8452", "Kkma": "#55A868"}

for name, counter in unk_counters.items():
    if not counter:
        print(f"{name}: UNK 단어 없음")
        continue

    top_items = counter.most_common(TOP_N)
    words  = [w for w, _ in top_items[::-1]]
    counts = [c for _, c in top_items[::-1]]

    fig, ax = plt.subplots(figsize=(8, max(4, len(words) * 0.35)))
    ax.barh(words, counts, color=colors.get(name, "gray"))
    ax.set_xlabel("등장 횟수")
    ax.set_title(f"{name} — UNK 단어 Top {TOP_N} (평가 데이터 기준)")
    plt.tight_layout()
    plt.savefig(f"figures/fig_unk_{name.lower()}.png", dpi=150)
    plt.show()